# 🚇 東京メトロ乗換シミュレーター：PyTorch 好み学習ノートブック（Colab対応）

このノートブックでは、Streamlitアプリで蓄積した選択ログ（`user_choices.jsonl`）を使って、
**「乗換を1回減らすためなら、何駅分の遠回りを許容するか」** といった経路の好みを PyTorch で初期状態から学習し、
学習前後で候補の並び順がどのように変わるかを検証します。

### 構成
1. データ読み込み・スキーマ確認・欠損／異常値チェック
2. 探索的データ分析（EDA：乗車区間数と乗換回数のトレードオフ可視化）
3. 学習・検証データ分割（データリーク防止のため分割後に標準化）
4. ベースライン vs PyTorch 線形効用モデル vs PyTorch 2層MLP の比較
5. 学習した好みの解釈（「乗換1回 ≒ 何駅分？」）と並び順の前後比較

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
np.random.seed(42)

FEATURE_NAMES = [
    "stops",
    "transfers",
    "diff_name_transfers",
    "transfer_timing",
]
print(f"PyTorch version: {torch.__version__}")

### 環境確認の結果
PyTorch・NumPy・Pandas・Matplotlib を読み込み、乱数シードを固定しました。特徴量はすべて東京メトロ全9路線に拡張した際もそのまま使える汎用4次元（乗車区間数・乗換回数・異駅名乗換回数・乗換タイミング）です。

In [ ]:
# 選択ログの読み込み（Colabの場合はアップロードも可能。無ければサンプルデータを自動生成）
LOG_FILE = Path("user_choices.jsonl")
records = []

if LOG_FILE.exists():
    with open(LOG_FILE, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    print(f"Loaded {len(records)} records from {LOG_FILE}")
else:
    print("user_choices.jsonl が見つからないため、デモ用サンプルログを使用します。")
    # ローカルアプリのログ形式と同一スキーマのサンプルデータ（乗換回避＋異駅名乗換控えめ傾向）
    records = [
        {"origin": "浅草", "destination": "新宿", "chosen_index": 0, "candidates": [
            {"title": "銀座経由", "stops": 18, "transfers": 1, "diff_name_transfers": 0, "transfer_timing": 0.0556, "features": [18.0, 1.0, 0.0, 0.0556]},
            {"title": "赤坂見附経由", "stops": 19, "transfers": 1, "diff_name_transfers": 0, "transfer_timing": 0.2368, "features": [19.0, 1.0, 0.0, 0.2368]},
            {"title": "溜池山王・国会議事堂前経由", "stops": 19, "transfers": 1, "diff_name_transfers": 1, "transfer_timing": 0.1842, "features": [19.0, 1.0, 1.0, 0.1842]},
        ]},
        {"origin": "渋谷", "destination": "銀座", "chosen_index": 0, "candidates": [
            {"title": "銀座線 直通", "stops": 8, "transfers": 0, "diff_name_transfers": 0, "transfer_timing": 0.0, "features": [8.0, 0.0, 0.0, 0.0]},
            {"title": "赤坂見附経由", "stops": 7, "transfers": 1, "diff_name_transfers": 0, "transfer_timing": 0.0714, "features": [7.0, 1.0, 0.0, 0.0714]},
            {"title": "溜池山王・国会議事堂前経由", "stops": 7, "transfers": 1, "diff_name_transfers": 1, "transfer_timing": 0.2143, "features": [7.0, 1.0, 1.0, 0.2143]},
        ]},
        {"origin": "青山一丁目", "destination": "銀座", "chosen_index": 0, "candidates": [
            {"title": "銀座線 直通", "stops": 5, "transfers": 0, "diff_name_transfers": 0, "transfer_timing": 0.0, "features": [5.0, 0.0, 0.0, 0.0]},
            {"title": "赤坂見附経由", "stops": 4, "transfers": 1, "diff_name_transfers": 0, "transfer_timing": -0.25, "features": [4.0, 1.0, 0.0, -0.25]},
            {"title": "溜池山王・国会議事堂前経由", "stops": 4, "transfers": 1, "diff_name_transfers": 1, "transfer_timing": 0.0, "features": [4.0, 1.0, 1.0, 0.0]},
        ]},
        {"origin": "表参道", "destination": "銀座", "chosen_index": 0, "candidates": [
            {"title": "銀座線 直通", "stops": 7, "transfers": 0, "diff_name_transfers": 0, "transfer_timing": 0.0, "features": [7.0, 0.0, 0.0, 0.0]},
            {"title": "赤坂見附経由", "stops": 6, "transfers": 1, "diff_name_transfers": 0, "transfer_timing": 0.0, "features": [6.0, 1.0, 0.0, 0.0]},
        ]},
        {"origin": "溜池山王", "destination": "銀座", "chosen_index": 0, "candidates": [
            {"title": "銀座線 直通", "stops": 3, "transfers": 0, "diff_name_transfers": 0, "transfer_timing": 0.0, "features": [3.0, 0.0, 0.0, 0.0]},
            {"title": "溜池山王・国会議事堂前経由", "stops": 2, "transfers": 1, "diff_name_transfers": 1, "transfer_timing": -0.5, "features": [2.0, 1.0, 1.0, -0.5]},
        ]},
        {"origin": "渋谷", "destination": "東京", "chosen_index": 0, "candidates": [
            {"title": "赤坂見附経由", "stops": 8, "transfers": 1, "diff_name_transfers": 0, "transfer_timing": 0.0, "features": [8.0, 1.0, 0.0, 0.0]},
            {"title": "溜池山王・国会議事堂前経由", "stops": 8, "transfers": 1, "diff_name_transfers": 1, "transfer_timing": 0.125, "features": [8.0, 1.0, 1.0, 0.125]},
            {"title": "銀座経由", "stops": 9, "transfers": 1, "diff_name_transfers": 0, "transfer_timing": 0.3889, "features": [9.0, 1.0, 0.0, 0.3889]},
        ]},
        {"origin": "外苑前", "destination": "霞ケ関", "chosen_index": 0, "candidates": [
            {"title": "赤坂見附経由", "stops": 4, "transfers": 1, "diff_name_transfers": 0, "transfer_timing": 0.0, "features": [4.0, 1.0, 0.0, 0.0]},
            {"title": "溜池山王・国会議事堂前経由", "stops": 4, "transfers": 1, "diff_name_transfers": 1, "transfer_timing": 0.25, "features": [4.0, 1.0, 1.0, 0.25]},
            {"title": "銀座経由", "stops": 7, "transfers": 1, "diff_name_transfers": 0, "transfer_timing": 0.3571, "features": [7.0, 1.0, 0.0, 0.3571]},
        ]},
        {"origin": "浅草", "destination": "新宿（銀座NG）", "chosen_index": 0, "candidates": [
            {"title": "赤坂見附経由", "stops": 19, "transfers": 1, "diff_name_transfers": 0, "transfer_timing": 0.2368, "features": [19.0, 1.0, 0.0, 0.2368]},
            {"title": "溜池山王・国会議事堂前経由", "stops": 19, "transfers": 1, "diff_name_transfers": 1, "transfer_timing": 0.1842, "features": [19.0, 1.0, 1.0, 0.1842]},
        ]},
        {"origin": "新宿", "destination": "銀座", "chosen_index": 0, "candidates": [
            {"title": "丸ノ内線 直通", "stops": 8, "transfers": 0, "diff_name_transfers": 0, "transfer_timing": 0.0, "features": [8.0, 0.0, 0.0, 0.0]},
            {"title": "赤坂見附経由", "stops": 9, "transfers": 1, "diff_name_transfers": 0, "transfer_timing": 0.0556, "features": [9.0, 1.0, 0.0, 0.0556]},
        ]},
        {"origin": "日本橋", "destination": "四ツ谷", "chosen_index": 0, "candidates": [
            {"title": "銀座経由", "stops": 6, "transfers": 1, "diff_name_transfers": 0, "transfer_timing": -0.1667, "features": [6.0, 1.0, 0.0, -0.1667]},
            {"title": "赤坂見附経由", "stops": 7, "transfers": 1, "diff_name_transfers": 0, "transfer_timing": 0.3571, "features": [7.0, 1.0, 0.0, 0.3571]},
            {"title": "溜池山王・国会議事堂前経由", "stops": 7, "transfers": 1, "diff_name_transfers": 1, "transfer_timing": 0.2143, "features": [7.0, 1.0, 1.0, 0.2143]},
        ]},
        {"origin": "上野", "destination": "新宿", "chosen_index": 0, "candidates": [
            {"title": "銀座経由", "stops": 15, "transfers": 1, "diff_name_transfers": 0, "transfer_timing": -0.0333, "features": [15.0, 1.0, 0.0, -0.0333]},
            {"title": "赤坂見附経由", "stops": 16, "transfers": 1, "diff_name_transfers": 0, "transfer_timing": 0.1875, "features": [16.0, 1.0, 0.0, 0.1875]},
        ]},
        {"origin": "東京", "destination": "渋谷", "chosen_index": 0, "candidates": [
            {"title": "赤坂見附経由", "stops": 8, "transfers": 1, "diff_name_transfers": 0, "transfer_timing": 0.0, "features": [8.0, 1.0, 0.0, 0.0]},
            {"title": "銀座経由", "stops": 9, "transfers": 1, "diff_name_transfers": 0, "transfer_timing": -0.3889, "features": [9.0, 1.0, 0.0, -0.3889]},
        ]},
    ]

# スキーマ・欠損・無効レコード検査
valid_records = []
invalid_count = 0
for r in records:
    cands = r.get("candidates", [])
    idx = r.get("chosen_index", -1)
    if len(cands) >= 2 and 0 <= idx < len(cands) and all(len(c.get("features", [])) == 4 for c in cands):
        valid_records.append(r)
    else:
        invalid_count += 1

print(f"有効イベント数: {len(valid_records)} / 除外（単一候補や欠損）: {invalid_count}")

### データ品質・スキーマ検査のまとめ
1候補しか存在しないイベント（比較対象がないため好みの情報を含まない）や、4次元特徴量・選択インデックスが欠損しているレコードを検査して除外しました。

In [ ]:
# EDA: 選ばれた候補 vs 選ばれなかった候補の特徴量分布を可視化
rows = []
for ev_id, r in enumerate(valid_records):
    for i, c in enumerate(r["candidates"]):
        rows.append({
            "event_id": ev_id,
            "route": f"{r['origin']}→{r['destination']}",
            "title": c["title"],
            "chosen": "選択" if i == r["chosen_index"] else "非選択",
            "stops": c["features"][0],
            "transfers": c["features"][1],
            "diff_name_transfers": c["features"][2],
            "transfer_timing": c["features"][3],
        })

df_eda = pd.DataFrame(rows)
print(df_eda.groupby("chosen")[FEATURE_NAMES].mean().round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for label, grp in df_eda.groupby("chosen"):
    axes[0].scatter(grp["stops"], grp["transfers"] + np.random.uniform(-0.06, 0.06, len(grp)), label=label, alpha=0.8, s=60)
axes[0].set_xlabel("Stops (ride segments)")
axes[0].set_ylabel("Transfers")
axes[0].set_title("Chosen vs Not Chosen: Stops vs Transfers")
axes[0].legend()
axes[0].grid(alpha=0.3)

df_eda.groupby("chosen")["diff_name_transfers"].mean().plot(kind="bar", ax=axes[1], color=["#2563EB", "#94A3B8"])
axes[1].set_title("Mean Different-Name Transfers Rate")
axes[1].set_ylabel("Rate")
axes[1].grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### EDAからの観察
- 選択された経路は非選択の経路に比べて **乗換回数** および **異駅名乗換の割合** が低い傾向が見られます。
- ただし、同じ乗換回数同士（例：浅草→新宿の銀座経由 vs 赤坂見附経由）では乗車区間数の短さも効いています。ここから、PyTorchモデルで両者のバランスを定量的に推定します。

In [ ]:
# Strict Featurization Ordering: まずイベント単位で Train / Val に分割してから前処理を行う
rng = np.random.default_rng(42)
perm = rng.permutation(len(valid_records))
split_idx = max(1, int(len(valid_records) * 0.75))
train_records = [valid_records[i] for i in perm[:split_idx]]
val_records = [valid_records[i] for i in perm[split_idx:]] or train_records[-2:]

print(f"Train events: {len(train_records)}, Validation events: {len(val_records)}")

def to_tensors(recs):
    return [
        (
            torch.tensor([c["features"] for c in r["candidates"]], dtype=torch.float32),
            torch.tensor(int(r["chosen_index"]), dtype=torch.long),
        )
        for r in recs
    ]

train_tensors = to_tensors(train_records)
val_tensors = to_tensors(val_records)

# MLP用にTrainのみで標準化パラメータを計算（ValにはTrainの統計量のみ適用）
all_train_x = torch.cat([x for x, _ in train_tensors], dim=0)
feat_mean = all_train_x.mean(dim=0)
feat_std = all_train_x.std(dim=0).clamp(min=1e-5)

# モデル1: 解釈可能な線形効用モデル（ゼロ初期化から学習）
class LinearChoiceModel(nn.Module):
    def __init__(self, in_features=4):
        super().__init__()
        self.linear = nn.Linear(in_features, 1, bias=False)
        nn.init.zeros_(self.linear.weight)

    def forward(self, x):
        return self.linear(x).squeeze(-1)

# モデル2: 非線形2層MLPモデル（表現力比較用）
class MLPChoiceModel(nn.Module):
    def __init__(self, in_features=4, hidden=12):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        x_norm = (x - feat_mean) / feat_std
        return self.net(x_norm).squeeze(-1)

def train_choice_model(model, train_data, val_data, epochs=250, lr=0.06, weight_decay=0.02):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    train_losses, val_losses = [], []
    for _ in range(epochs):
        model.train()
        opt.zero_grad()
        loss = sum(F.cross_entropy(model(x).unsqueeze(0), y.unsqueeze(0)) for x, y in train_data) / len(train_data)
        loss.backward()
        opt.step()
        train_losses.append(float(loss.item()))

        model.eval()
        with torch.no_grad():
            vloss = sum(F.cross_entropy(model(x).unsqueeze(0), y.unsqueeze(0)) for x, y in val_data) / len(val_data)
            val_losses.append(float(vloss.item()))
    return train_losses, val_losses

def eval_top1(model, data):
    model.eval()
    correct = 0
    with torch.no_grad():
        for x, y in data:
            if int(torch.argmax(model(x)).item()) == int(y.item()):
                correct += 1
    return correct / len(data)

# ベースライン評価: 駅数最短順
def eval_shortest_stops_baseline(recs):
    correct = 0
    for r in recs:
        pred = min(range(len(r["candidates"])), key=lambda i: (r["candidates"][i]["features"][0], r["candidates"][i]["features"][1]))
        if pred == r["chosen_index"]:
            correct += 1
    return correct / len(recs)

linear_model = LinearChoiceModel()
lin_tr, lin_va = train_choice_model(linear_model, train_tensors, val_tensors, epochs=250, lr=0.08, weight_decay=0.02)

mlp_model = MLPChoiceModel()
mlp_tr, mlp_va = train_choice_model(mlp_model, train_tensors, val_tensors, epochs=250, lr=0.03, weight_decay=0.03)

comparison = pd.DataFrame([
    {"Model": "Baseline (駅数最短順)", "Train Top-1": eval_shortest_stops_baseline(train_records), "Val Top-1": eval_shortest_stops_baseline(val_records), "Final Val Loss": np.nan},
    {"Model": "PyTorch Linear Choice", "Train Top-1": eval_top1(linear_model, train_tensors), "Val Top-1": eval_top1(linear_model, val_tensors), "Final Val Loss": round(lin_va[-1], 4)},
    {"Model": "PyTorch 2-Layer MLP", "Train Top-1": eval_top1(mlp_model, train_tensors), "Val Top-1": eval_top1(mlp_model, val_tensors), "Final Val Loss": round(mlp_va[-1], 4)},
])
print(comparison.to_string(index=False))

plt.figure(figsize=(8, 4))
plt.plot(lin_tr, label="Linear Train Loss")
plt.plot(lin_va, "--", label="Linear Val Loss")
plt.plot(mlp_tr, label="MLP Train Loss")
plt.plot(mlp_va, "--", label="MLP Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Listwise Cross-Entropy Loss")
plt.title("Training & Validation Loss Curves")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### モデル比較の考察
- **Baseline（駅数最短順）** は「渋谷→銀座」のように1区間短い代わりに1回乗り換える経路を選んでしまうため、乗換回避派の好みとはズレが生じます。
- **PyTorch Linear Choice モデル** はパラメータ数が4個と少なく少量データでも過学習しにくい上、重みの比から **「乗換1回 ≒ 何駅分の遠回り許容か」** を直接読み取れるため、本アプリの本番再ランキングモデルとして最適です。
- ログが数百件以上に増えた段階では、非線形な相互作用（例：長距離時だけ乗換を許容するなど）を捉えられる **2層MLP** との比較がさらに有効になります。

In [ ]:
# 学習した重みの可視化と「乗換1回あたりの許容遠回り駅数」の算出
w = linear_model.linear.weight.detach().numpy().flatten()
for name, val in zip(FEATURE_NAMES, w):
    print(f"{name:22s}: {val:+.4f}")

if w[0] < 0 and w[1] < 0:
    eq_stops = w[1] / w[0]
    print(f"\n👉 解釈：乗換を1回減らすためなら、約 {eq_stops:.2f} 駅分までの遠回りを許容する傾向です。")

# 学習前後の並び順変化を確認
print("\n=== 並び順の変化（駅数最短順 vs PyTorch学習後）===")
for r in valid_records[:5]:
    cands = r["candidates"]
    before_order = sorted(cands, key=lambda c: (c["features"][0], c["features"][1]))
    with torch.no_grad():
        scores = linear_model(torch.tensor([c["features"] for c in cands], dtype=torch.float32)).tolist()
    after_order = [c for _, c in sorted(zip(scores, cands), key=lambda p: -p[0])]
    print(f"{r['origin']} → {r['destination']}:")
    print(f"  変更前(駅数順): {[c['title'] for c in before_order]}")
    print(f"  学習後(好み順): {[c['title'] for c in after_order]}")

### まとめ
1. **データ表現**: 駅×路線ノードのグラフから抽出した4特徴量により、少量ログでも意味のある好みを学習できました。
2. **定量解釈**: 線形効用モデルの重み比 `w_transfers / w_stops` から、乗換1回に対する許容遠回り駅数を直感的に確認できました。
3. **今後の拡張**: 東京メトロ全9路線へ拡大しても同じ特徴量スキーマのまま学習でき、さらに時刻表データ（所要時間・乗換徒歩分）を取り込む段階で特徴量を追加するだけで同じ枠組みがそのまま機能します。